# Tensors

Everything in PyTorch is a tensor: a rectangular block of numbers with a shape and a dtype.
A scalar, a vector, a matrix and an image batch are the same kind of object at different
ranks.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

scalar = torch.tensor(7.5)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
tensor = torch.zeros(2, 3, 4)

for name, t in (("scalar", scalar), ("vector", vector), ("matrix", matrix), ("tensor", tensor)):
    print(f"{name:7} shape {str(tuple(t.shape)):12} ndim {t.ndim}  dtype {t.dtype}")

### A matrix can be represented as an image

Thus a matrix can also be drawn

In [ ]:
x = torch.arange(48, dtype=torch.float32).reshape(6, 8)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].imshow(x, cmap="viridis")
axes[0].set_title(f"x  {tuple(x.shape)}")

axes[1].imshow(x.T, cmap="viridis")
axes[1].set_title(f"x.T  {tuple(x.T.shape)}")

plt.tight_layout()
plt.show()

### The samples from an image dataset are three matrices

PyTorch stores images as `(channels, height, width)`.

For Matplotlib we use `(height, width, channels)`

In [ ]:
h = w = 64
ramp = torch.linspace(0, 1, h)

red = ramp[None, :].repeat(h, 1)
green = ramp[:, None].repeat(1, w)
blue = torch.full((h, w), 0.45)

img = torch.stack([red, green, blue])

fig, axes = plt.subplots(1, 4, figsize=(11, 2.9))

for ax, ch, name in zip(axes[:3], img, ("red", "green", "blue")):
    ax.imshow(ch, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis("off")

axes[3].imshow(img.permute(1, 2, 0))
axes[3].set_title(f"permute -> {tuple(img.permute(1, 2, 0).shape)}")
axes[3].axis("off")

plt.tight_layout()
plt.show()

### Broadcasting

Shapes are aligned.

In [ ]:
a = torch.tensor([[1.0], [2.0], [3.0]])
b = torch.tensor([[10.0, 20.0, 30.0, 40.0]])
c = a + b

print("a", tuple(a.shape), " b", tuple(b.shape), "->  c", tuple(c.shape))

fig, axes = plt.subplots(1, 3, figsize=(10, 2.7))

for ax, m, title in zip(axes, (a, b, c), ("a", "b", "a + b")):
    ax.imshow(m, cmap="Blues", vmin=0, vmax=float(c.max()))
    for (i, j), v in np.ndenumerate(m.numpy()):
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{title}  {tuple(m.shape)}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

### Batched matrix multiplication

A batch of matrices is a 3-D tensor. `@` applies the multiplication to every batch element
independently, which is why one line handles a whole mini-batch.

In [ ]:
A = torch.rand(8, 4, 5)
B = torch.rand(5, 3)

C = A @ B

print(tuple(A.shape), "@", tuple(B.shape), "->", tuple(C.shape))

print ("A_0:")
print (A[0])
print ("C_0:")
print (C[0])
print ("Multiplication with single matrix:")
print (A[0] @ B)

try:
    torch.rand(3, 4) @ torch.rand(3, 4)
except RuntimeError as e:
    print("\nmismatch:", e)

### Very large batched matrix multiplication

Same multiplication done one thousand times, once on the CPU and once on the GPU.

8 matrices of 512 x 512, each multiplied by a 512 x 512 matrix.

In [ ]:
import time

BATCH, N, REPS = 8, 512, 5000


def bench(device):
    A = torch.randn(BATCH, N, N, device=device)
    B = torch.randn(N, N, device=device)

    A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(REPS):
        A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    dt = time.perf_counter() - t0
    return dt, 2 * BATCH * N ** 3 * REPS / dt / 1e9


devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))
else:
    print("no GPU on this machine or cuda/driver mismatch\n")

print(f"{BATCH} x ({N} x {N}) @ ({N} x {N}), {REPS} repetitions")
print(f"A holds {BATCH * N * N / 1e6:.1f} M numbers ({BATCH * N * N * 4 / 1e6:.0f} MB)")
print(f"threads: {torch.get_num_threads()}\n")

timing = {}
for device in devices:
    seconds, gflops = bench(device)
    timing[device.type] = seconds
    print(f"{device.type:4} {seconds:8.2f} s   {seconds / REPS * 1000:7.3f} ms each   {gflops:8.1f} GFLOP/s")

if "cuda" in timing:
    print(f"\ncuda is {timing['cpu'] / timing['cuda']:.0f}x faster than cpu")

In [ ]:
import time

BATCH, N, REPS = 8, 512, 10000


def bench(device):
    A = torch.randn(BATCH, N, N, device=device)
    B = torch.randn(N, N, device=device)

    A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(REPS):
        A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    dt = time.perf_counter() - t0
    return dt, 2 * BATCH * N ** 3 * REPS / dt / 1e9


devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))
else:
    print("no GPU on this machine or cuda/driver mismatch\n")

print(f"{BATCH} x ({N} x {N}) @ ({N} x {N}), {REPS} repetitions")
print(f"A holds {BATCH * N * N / 1e6:.1f} M numbers ({BATCH * N * N * 4 / 1e6:.0f} MB)")
print(f"threads: {torch.get_num_threads()}\n")

timing = {}
for device in devices:
    seconds, gflops = bench(device)
    timing[device.type] = seconds
    print(f"{device.type:4} {seconds:8.2f} s   {seconds / REPS * 1000:7.3f} ms each   {gflops:8.1f} GFLOP/s")

if "cuda" in timing:
    print(f"\ncuda is {timing['cpu'] / timing['cuda']:.0f}x faster than cpu")

In [ ]:
import time

BATCH, N, REPS = 16, 512, 10000


def bench(device):
    A = torch.randn(BATCH, N, N, device=device)
    B = torch.randn(N, N, device=device)

    A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(REPS):
        A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    dt = time.perf_counter() - t0
    return dt, 2 * BATCH * N ** 3 * REPS / dt / 1e9


devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))
else:
    print("no GPU on this machine or cuda/driver mismatch\n")

print(f"{BATCH} x ({N} x {N}) @ ({N} x {N}), {REPS} repetitions")
print(f"A holds {BATCH * N * N / 1e6:.1f} M numbers ({BATCH * N * N * 4 / 1e6:.0f} MB)")
print(f"threads: {torch.get_num_threads()}\n")

timing = {}
for device in devices:
    seconds, gflops = bench(device)
    timing[device.type] = seconds
    print(f"{device.type:4} {seconds:8.2f} s   {seconds / REPS * 1000:7.3f} ms each   {gflops:8.1f} GFLOP/s")

if "cuda" in timing:
    print(f"\ncuda is {timing['cpu'] / timing['cuda']:.0f}x faster than cpu")

### dtype and device

Always check PyTorch shape and dtype if run into errors and a third common error that can happen is
putting tensors on different devices.

In [ ]:
print("float default :", torch.tensor([1.0]).dtype)
print("int default   :", torch.tensor([1]).dtype)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand(2, 3).to(device)

print("device        :", x.device)
print("float64 copy  :", x.to(torch.float64).dtype)